# ETL Fase 3: Enriquecimiento de Datos mediante Web Scraping
Este tercer notebook tiene como objetivo expandir nuestro conjunto de datos original extrayendo información adicional de forma automatizada directamente desde internet. A este proceso se le conoce como **Web Scraping**. El objetivo de negocio aquí es identificar la huella operativa global de cada franquicia (es decir, en qué otros videojuegos tienen divisiones activas), lo cual nos permitirá medir su nivel de diversificación y atractivo financiero.

## 1. Importación de Librerías y Herramientas de Extracción
Para lograr esto, necesitamos que Python sea capaz de comportarse como un navegador web: debe conectarse a una página, descargar su código fuente y extraer únicamente los textos que nos interesan. Además, incorporaremos herramientas para realizar un "Scraping Ético", simulando el comportamiento humano para no saturar los servidores de destino.

In [ ]:
# Herramientas estándar para manejo de rutas y creación de DataFrames
import os
import pandas as pd

# 'requests' es la librería que le permite a Python enviar peticiones a internet. 
# Funciona igual que cuando escribes una dirección web (URL) en Chrome y presionas 'Enter'.
import requests

# 'BeautifulSoup' es nuestra herramienta de disección. 
# Toma el código fuente (HTML) crudo y desordenado de una página web y nos permite navegar por él 
# buscando etiquetas específicas (como tablas, párrafos o listas).
from bs4 import BeautifulSoup

# Librerías para el control del tiempo y la aleatoriedad.
# Son fundamentales para el "Scraping Ético": nos permiten pausar el código por unos segundos 
# entre cada petición web, evitando saturar el servidor destino o ser bloqueados por parecer un ataque automatizado (bot).
import time
import random

# Mensaje de confirmación visual
print("Librerías de Web Scraping importadas correctamente. El entorno está listo para extraer datos de la web.")

Librerías de Web Scraping importadas correctamente.


## 2. Construcción del Robot Extractor (Scraper)
Para extraer los datos de Wikipedia de forma ordenada y profesional, diseñamos un "Scraper" orientado a objetos. Imagina que estamos entrenando a un asistente virtual para que haga el trabajo manual por nosotros. Le enseñamos a seguir esta estrategia de tres pasos:

1. **Fase de Exploración:** Navega a la página principal del campeonato, busca el listado de todos los clubes participantes y anota la dirección web (URL) del perfil de Wikipedia de cada uno. Inteligente mente, ignora los enlaces "rotos" (clubes que aún no tienen página propia).
2. **Fase de Extracción:** Visita una por una las páginas de los clubes anotados. Busca la "Ficha Técnica" (Infobox) lateral, localiza la sección de "Divisiones" y extrae los nombres de los videojuegos donde el club tiene equipos activos, limpiando anotaciones extrañas en el texto.
3. **Scraping Ético (Control de Tráfico):** Para no saturar los servidores de Wikipedia ni ser detectados como un ataque malicioso, el robot hace pausas aleatorias de un par de segundos entre cada visita. Finalmente, consolida todo en una tabla y lo guarda en nuestro disco duro.

In [ ]:
class WikiDivisionsScraper:
    def __init__(self, main_url):
        """Constructor: Aquí preparamos la 'mochila' de nuestro robot antes de salir a explorar."""
        self.main_url = main_url
        self.base_wiki_url = "https://en.wikipedia.org"
        
        # 'Headers': Es un disfraz. Le decimos a Wikipedia que somos un humano usando Chrome en Windows, 
        # para que no nos bloquee por ser un script automatizado.
        self.headers = {
            "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
        }
        
        # Listas vacías donde el robot guardará lo que vaya encontrando
        self.clubes_a_procesar = [] # Guardará tuplas de (Nombre_Club, URL_Club)
        self.resultados_divisiones = [] # Guardará los datos finales para convertirlos en tabla

    def obtener_lista_clubes(self):
        """Fase 1: Extrae los nombres de los clubes y sus URLs desde la página principal del evento."""
        try:
            print(f"🌐 Conectando a la página principal de la EWC...")
            # Hacemos la petición a la página y le damos 10 segundos máximo para responder
            response = requests.get(self.main_url, headers=self.headers, timeout=10)
            response.raise_for_status() # Verifica que la página cargó bien (Error 200 OK)
            
            # BeautifulSoup convierte el texto de la web en una estructura navegable
            soup = BeautifulSoup(response.text, 'html.parser')
            
            # Buscamos la caja HTML específica donde están listados los equipos
            contenedor_columnas = soup.find('div', class_='columns-5')
            
            if not contenedor_columnas:
                # Plan B: Por si Wikipedia cambió sutilmente el diseño de la página
                contenedor_columnas = soup.find('div', class_='columns-start')
                
            if contenedor_columnas:
                # Extraemos todas las viñetas (etiquetas <li> de lista)
                elementos_li = contenedor_columnas.find_all('li')
                for li in elementos_li:
                    enlace = li.find('a') # Buscamos el link (etiqueta <a>) dentro de la viñeta
                    if enlace:
                        nombre_club = enlace.text.strip()
                        
                        # TRUCO DE CALIDAD: Los enlaces con clase 'new' en Wikipedia son "redlinks" (páginas que no existen)
                        if enlace.has_attr('class') and 'new' in enlace['class']:
                            print(f"⏩ Saltando '{nombre_club}' porque no tiene página propia en Wikipedia.")
                            continue
                            
                        # Construimos la dirección web completa del club
                        url_relativa = enlace['href']
                        url_absoluta = self.base_wiki_url + url_relativa
                        
                        # Guardamos el club en nuestra lista de tareas
                        self.clubes_a_procesar.append((nombre_club, url_absoluta))
                        
                print(f"✅ Se encontraron {len(self.clubes_a_procesar)} clubes válidos listos para ser raspados.\n")
            else:
                print("❌ No se pudo encontrar el contenedor principal de los equipos.")
                
        except requests.exceptions.RequestException as e:
            # Capturamos fallos de internet sin que el programa "explote"
            print(f"❌ Error al conectar a la página principal: {e}")

    def raspar_divisiones_club(self, nombre_club, url_club):
        """Fase 2: Visita el perfil del club, lee su ficha técnica y extrae los videojuegos."""
        try:
            print(f"🔍 Buscando divisiones de: {nombre_club}...")
            response = requests.get(url_club, headers=self.headers, timeout=10)
            response.raise_for_status()
            
            soup = BeautifulSoup(response.text, 'html.parser')
            # Localizamos la 'infobox' (la tabla resumen que suele estar a la derecha en Wikipedia)
            infobox = soup.find('table', class_='infobox')
            
            if not infobox:
                print(f"⚠️ No se encontró la ficha técnica en la página de {nombre_club}.")
                return
                
            # Buscamos la fila exacta que tenga el título "Divisions"
            fila_divisiones = None
            filas = infobox.find_all('tr')
            for fila in filas:
                encabezado = fila.find('th', class_='infobox-label')
                if encabezado and 'divisions' in encabezado.text.lower():
                    fila_divisiones = fila
                    break
                    
            if fila_divisiones:
                datos_celda = fila_divisiones.find('td', class_='infobox-data')
                if datos_celda:
                    # Extraemos todos los elementos de la lista de juegos
                    juegos_li = datos_celda.find_all('li')
                    if juegos_li:
                        for item in juegos_li:
                            nombre_juego = item.text.strip()
                            
                            # Limpieza: Si el juego dice "League of Legends (equipo academia)", cortamos desde el paréntesis
                            if '(' in nombre_juego:
                                nombre_juego = nombre_juego.split('(')[0].strip()
                                
                            # Guardamos la relación Club-Juego en nuestro diccionario maestro
                            self.resultados_divisiones.append({
                                'Organization_Name': nombre_club,
                                'Game_Title': nombre_juego
                            })
                    else:
                        # Si no hay una lista formal, alertamos que está en formato de texto plano
                        texto_plano = datos_celda.text.strip()
                        print(f"ℹ️ Formato alternativo detectado para {nombre_club}: {texto_plano}")
            else:
                print(f"⚠️ El club {nombre_club} no tiene una sección de 'Divisions' en su perfil.")
                
        except requests.exceptions.RequestException as e:
            print(f"❌ Error al conectar a la página de {nombre_club}: {e}")

    def ejecutar_pipeline_completo(self):
        """Orquestador Principal: Dirige a nuestro robot para que ejecute los pasos en orden."""
        # Paso 1: Obtener la lista de URLs a visitar
        self.obtener_lista_clubes()
        
        # Paso 2: Recorrer cada club extrayendo su información
        for i, (nombre, url) in enumerate(self.clubes_a_procesar, 1):
            self.raspar_divisiones_club(nombre, url)
            
            # CONTROL ÉTICO: Hacemos que el programa se detenga entre 1.5 y 3 segundos
            # Esto evita baneos automáticos de la IP y respeta los servidores de Wikipedia.
            if i < len(self.clubes_a_procesar): 
                tiempo_espera = random.uniform(1.5, 3.0)
                print(f"⏳ Esperando {tiempo_espera:.2f} segundos antes del siguiente club... [{i}/{len(self.clubes_a_procesar)}]")
                time.sleep(tiempo_espera)
                
        # Paso 3: Convertir los resultados en una tabla de Pandas (DataFrame)
        df_final = pd.DataFrame(self.resultados_divisiones)
        
        if not df_final.empty:
            print(f"\n🎉 ¡Proceso finalizado! Se extrajeron {len(df_final)} relaciones Club-División.")
            self._guardar_en_csv(df_final)
            return df_final
        else:
            print("❌ No se pudo extraer ninguna información.")
            return pd.DataFrame()

    def _guardar_en_csv(self, df):
        """Paso final: Guarda la tabla extraída en nuestra carpeta de datos procesados."""
        ruta_carpeta = "data/processed"
        # Nos aseguramos de que la carpeta exista. Si no, Python la crea automáticamente.
        os.makedirs(ruta_carpeta, exist_ok=True)
        
        # Unimos la ruta y exportamos el archivo
        ruta_archivo = os.path.join(ruta_carpeta, "club_divisions_scraped.csv")
        df.to_csv(ruta_archivo, index=False, encoding='utf-8')
        print(f"💾 DataFrame exportado exitosamente a: {ruta_archivo}")

# ==========================================
# 3. EJECUCIÓN DEL PROCESO
# ==========================================
# Definimos la URL de inicio y "encendemos" nuestro robot
url_mundial = 'https://en.wikipedia.org/wiki/2025_Esports_World_Cup'

scraper = WikiDivisionsScraper(main_url=url_mundial)
df_club_divisions = scraper.ejecutar_pipeline_completo()

# Mostramos las primeras 10 filas de lo que el robot logró extraer
df_club_divisions.head(10)

🌐 Conectando a la página principal de la EWC...
⏩ Saltando 'Gentle Mates' porque no tiene página propia en Wikipedia (Redlink).
⏩ Saltando 'Team BDS' porque no tiene página propia en Wikipedia (Redlink).
⏩ Saltando 'Gaimin Gladiators' porque no tiene página propia en Wikipedia (Redlink).
⏩ Saltando 'All Gamers' porque no tiene página propia en Wikipedia (Redlink).
⏩ Saltando 'ZETA DIVISION' porque no tiene página propia en Wikipedia (Redlink).
⏩ Saltando 'Leviatán' porque no tiene página propia en Wikipedia (Redlink).
⏩ Saltando 'POWR Esports' porque no tiene página propia en Wikipedia (Redlink).
✅ Se encontraron 33 clubes válidos listos para ser raspados.

🔍 Buscando divisiones de: Fnatic...
⏳ Esperando 2.55 segundos antes del siguiente club... [1/33]
🔍 Buscando divisiones de: G2 Esports...
⏳ Esperando 1.98 segundos antes del siguiente club... [2/33]
🔍 Buscando divisiones de: HEROIC...
ℹ️ Formato alternativo detectado para HEROIC: Counter-Strike 2Dota 2
⏳ Esperando 2.20 segundos antes

,Organization_Name,Game_Title
0,Fnatic,Apex Legends
1,Fnatic,Counter-Strike 2
2,Fnatic,League of Legends
3,Fnatic,Overwatch 2
4,Fnatic,Street Fighter
5,Fnatic,Rainbow Six Siege
6,Fnatic,Valorant
7,Fnatic,The Finals
8,G2 Esports,Counter-Strike 2
9,G2 Esports,EVA


## 3. Almacenamiento Local de los Datos Raspados (Data Persistence)
Una vez que nuestro robot termina de extraer la información de internet, el resultado queda guardado únicamente en la memoria RAM de la computadora. Si el notebook se cierra, perderíamos todo el trabajo y tendríamos que volver a conectarnos a Wikipedia (lo cual consume tiempo y recursos). Por esta razón, el siguiente paso consiste en persistir la información, creando automáticamente una estructura de carpetas en nuestro proyecto (`data/processed`) y exportando el DataFrame a un archivo físico `.csv`. Configurar correctamente el parámetro de codificación `utf-8` es un requisito indispensable para garantizar que los caracteres especiales de los nombres de los clubes extranjeros no se rompan ni se corrompan en el proceso.

In [ ]:
# 1. Definir la ruta de la carpeta y el nombre del archivo
# Usamos rutas relativas de dos puntos ('../') para subir un nivel en el proyecto y 
# apuntar directamente a la carpeta de datos procesados.
ruta_carpeta = "../data/processed"
ruta_archivo = os.path.join(ruta_carpeta, "club_divisions_scraped.csv")

# 2. Crear la carpeta física en el sistema operativo si no existe
# El parámetro 'exist_ok=True' actúa como un seguro: si la carpeta ya existe, el script continúa 
# sin arrojar ningún error; si no existe, la crea en ese mismo instante.
os.makedirs(ruta_carpeta, exist_ok=True)

# 3. Guardar el DataFrame en formato CSV
# index=False: Indica a pandas que descarte la columna del índice numérico implícito para no ensuciar el CSV.
# encoding='utf-8': Es CRUCIAL en ciencia de datos; asegura que los caracteres especiales, eñes, 
# tildes y alfabetos internacionales de los nombres de los equipos se guarden correctamente.
df_club_divisions.to_csv(ruta_archivo, index=False, encoding='utf-8')

# 4. Mensaje de confirmación de fin de proceso
print(f"✅ ¡Excelente! El archivo intermedio ha sido guardado con éxito en: {ruta_archivo}")

✅ ¡Excelente! El archivo ha sido guardado con éxito en: ../data/processed\club_divisions_scraped.csv


## 4. Configuración del Entorno para la Carga en la Base de Datos
Una vez que hemos completado la extracción masiva desde internet y respaldado los datos en un archivo CSV local, el siguiente objetivo es almacenar esta información en nuestro servidor relacional de SQL Server. Para lograrlo, iniciamos la fase de preparación de la conexión importando los módulos necesarios para formatear cadenas de conexión seguras y gestionar las credenciales del sistema a través de nuestras variables de entorno.

In [ ]:
# Módulos para parsear cadenas de texto especiales y construir la URL de conexión de la BD
import urllib.parse
from sqlalchemy import create_engine

# Herramientas para localizar y cargar de forma segura las credenciales ocultas en el archivo .env
from dotenv import load_dotenv, find_dotenv

# Módulo del sistema para manipular las rutas de búsqueda de Python
import sys
import os # Aseguramos la importación de os para evitar errores con abspath

# Agregamos la ruta raíz del proyecto al sistema.
# Esto nos permite que este notebook de Web Scraping comparta la misma configuración 
# y variables de entorno del resto del pipeline sin duplicar archivos.
sys.path.append(os.path.abspath('..'))

# Mensaje de confirmación de carga de librerías de infraestructura de datos
print("Librerías de base de datos importadas correctamente para cargar el enriquecimiento.")

Librerías importadas correctamente para las tablas de detalle.


## 5. Autenticación y Conexión al Servidor Relacional
Con nuestros datos externos ya estructurados en un DataFrame, el paso final de este sub-flujo es conectarnos a la base de datos para preparar la inserción. Volvemos a invocar el archivo de configuración `.env` para extraer las variables del servidor y el driver ODBC de forma segura. Al igual que en los procesos de ingesta previos, utilizaremos la autenticación integrada de Windows (`Trusted_Connection=yes`) empaquetada a través de SQLAlchemy, garantizando que el puente hacia SQL Server sea idéntico y apunte a la misma base de datos destino.

In [ ]:
# 1. Cargar las variables de entorno desde el archivo seguro .env
load_dotenv(find_dotenv())

# Extraemos los parámetros de conexión sin exponer nombres de servidores reales en el código fuente
server = os.getenv('DB_SERVER')
database = os.getenv('DB_DATABASE')
driver = os.getenv('DB_DRIVER')

# 2. Control de Incolumidad: Validamos que el entorno reconozca las credenciales necesarias
# Si se ejecuta el script fuera de la estructura de carpetas correcta, este bloque frenará el proceso de inmediato.
if not server or not database:
    raise ValueError("Error Crítico: No se encontraron las credenciales de la BD en el archivo .env")

# 3. Formateo y enmascaramiento de la cadena de conexión
# 'urllib.parse.quote_plus' se encarga de codificar caracteres especiales (como espacios o llaves) 
# para que la URL de conexión de red no se rompa al comunicarse con el driver de SQL Server.
params = urllib.parse.quote_plus(
    f"DRIVER={{{driver}}};SERVER={server};DATABASE={database};Trusted_Connection=yes;"
)

# 4. Instanciación del motor de persistencia (Engine) de SQLAlchemy
# Este objeto administrará el pool de conexiones y traducirá nuestras peticiones de pandas al dialecto de T-SQL.
engine = create_engine(f"mssql+pyodbc:///?odbc_connect={params}")

# Mensaje de confirmación en la consola del pipeline
print("Motor de base de datos configurado exitosamente. Todo listo para inyectar el catálogo enriquecido.")

Motor de base de datos configurado y listo para insertar detalles.


## 6. Homologación y Validación Cruzada de Integridad (Transform)
Los datos extraídos mediante Web Scraping provienen de fuentes abiertas y colaborativas (como Wikipedia), lo que significa que con frecuencia sufren de inconsistencias de nomenclatura. Por ejemplo, mientras nuestra base de datos registra de forma precisa `PUBG: Battlegrounds`, Wikipedia puede listarlo simplemente como `PUBG`. 

Para resolver este choque de formatos y evitar violaciones de integridad referencial, este bloque ejecuta una estrategia de **Doble Validación Cruzada**:
1. **Limpieza Estándar:** Removemos espacios invisibles residuales en los nombres de clubes y videojuegos.
2. **Diccionario de Homologación:** Mapeamos y traducimos de forma masiva los títulos genéricos de Wikipedia a las convenciones exactas de nuestra base de datos.
3. **Validación contra Producción:** Consultamos en tiempo real las tablas maestras de SQL Server (`Clubs` y `Tournaments`) y extraemos los catálogos válidos. 
4. **Filtro de Intersección:** Aplicamos un filtro lógico estricto (`AND`). Solo los registros que existan simultáneamente en el catálogo de clubes y en el de torneos sobrevivirán, descartando duplicados y registros huérfanos.

In [ ]:
# 1. LIMPIEZA DE ESPACIOS INVISIBLES
# Eliminamos espacios en blanco accidentales al inicio o final para evitar fallos de coincidencia por texto sutil.
df_club_divisions['Organization_Name'] = df_club_divisions['Organization_Name'].str.strip()
df_club_divisions['Game_Title'] = df_club_divisions['Game_Title'].str.strip()

# 2. HOMOLOGACIÓN DE DATOS (Diccionario de Traducción)
# Conectamos las discrepancias lingüísticas de Wikipedia (izquierda) con las llaves oficiales de nuestra BD (derecha).
mapeo_juegos = {
    'Street Fighter': 'Street Fighter 6',
    'EA Sports FC': 'EA Sports FC 25',
    'Tom Clancy\'s Rainbow Six Siege': 'Rainbow Six Siege X',
    'Crossfire': 'CrossFire',
    'Rainbow Six Siege': 'Rainbow Six Siege X',
    'Call of Duty': 'Call of Duty: Warzone',
    'PUBG': 'PUBG: Battlegrounds'
}

# Aplicamos la traducción masiva sobre la columna de videojuegos
df_club_divisions['Game_Title'] = df_club_divisions['Game_Title'].replace(mapeo_juegos)

# 3. VALIDACIÓN DE LLAVE FORÁNEA CONTRA SQL SERVER (Descarga de Catálogos de Control)
# Extraemos los nombres de clubes válidos directamente desde nuestra base de datos relacional
organizacion_bd_df = pd.read_sql("SELECT Organization_Name FROM Clubs", con=engine)
organizaciones_validas = organizacion_bd_df['Organization_Name'].tolist()

# Extraemos los títulos de videojuegos autorizados directamente desde la tabla Tournaments
juegos_bd_df = pd.read_sql("SELECT Game_Title FROM Tournaments", con=engine)
juegos_validos = juegos_bd_df['Game_Title'].tolist()

# 4. AUDITORÍA DE REGISTROS HUÉRFANOS
# Identificamos clubes extraídos de la web que no existen en nuestra tabla maestra 'Clubs'
organizaciones_huerfanos = df_club_divisions[~df_club_divisions['Organization_Name'].isin(organizaciones_validas)]['Organization_Name'].unique()
if len(organizaciones_huerfanos) > 0:
    print(f"⚠️ ADVERTENCIA: Se detectaron clubes de Wikipedia no registrados en BD (se excluirán): {organizaciones_huerfanos}")
else:
    print("✅ ¡Perfecto! Todas las organizaciones de Wikipedia coinciden con la Base de Datos.")

# Identificamos videojuegos de la web que no corresponden a ningún torneo oficial registrado
juegos_huerfanos = df_club_divisions[~df_club_divisions['Game_Title'].isin(juegos_validos)]['Game_Title'].unique()
if len(juegos_huerfanos) > 0:
    print(f"⚠️ ADVERTENCIA: Se detectaron juegos en Wikipedia no registrados en BD (se excluirán): {juegos_huerfanos}")
else:
    print("✅ ¡Perfecto! Todos los juegos del Scraping coinciden con la Base de Datos.")

# 5. FILTRADO FINAL DE SEGURIDAD Y DE-DUPLICACIÓN
# Con el operador '&' (AND lógico) aseguramos que tanto el Club como el Juego existan previamente en SQL Server.
# Aplicamos '.drop_duplicates()' para limpiar cualquier redundancia generada durante el raspado de datos.
df_insertar = df_club_divisions[
    df_club_divisions['Organization_Name'].isin(organizaciones_validas) & 
    df_club_divisions['Game_Title'].isin(juegos_validos)
].drop_duplicates()

# 6. Control de Volumetría Final
print(f"\nTotal de divisiones integradas con éxito y listas para inserción segura: {len(df_insertar)}")

# Desplegamos la estructura limpia final
df_insertar.head(5)

✅ ¡Perfecto! Todos las organizaciones del Roster coinciden con la Base de Datos.
⚠️ ADVERTENCIA: Aún se excluirán estos juegos: <StringArray>
[                  'The Finals',                          'EVA',
                     'Fortnite',                      'iRacing',
                  'Hearthstone',                 'Kings League',
 'Call of Duty: Black Ops 7[a]',       'Pokémon Scarlet/Violet',
                'Marvel Rivals',                         'Halo',
 'League of Legends: Wild Rift',      'Super Smash Bros. Melee',
            'World of Warcraft',                    'Overwatch',
               'Arena of Valor',                  'Brawl Stars',
                   'Identity V',            'Super Smash Bros.',
                         'FIFA']
Length: 19, dtype: str
Total de divisiones integrados con éxito: 69


,Organization_Name,Game_Title
0,Fnatic,Apex Legends
1,Fnatic,Counter-Strike 2
2,Fnatic,League of Legends
3,Fnatic,Overwatch 2
4,Fnatic,Street Fighter 6


## 7. Ingesta del Catálogo Enriquecido en SQL Server (Load)
Este último bloque representa la fase de carga de nuestro sub-flujo de Web Scraping. Después de haber extraído la información de internet de manera ética, respaldarla localmente en un archivo CSV, limpiar sus inconsistencias tipográficas y validar su integridad contra los catálogos existentes en producción, el DataFrame `df_insertar` se encuentra 100% purificado. 

Para su almacenamiento definitivo, ejecutamos la función de carga segura `cargar_tabla_detalle`. Esta operación insertará los registros directamente en la tabla transaccional `Club_Divisions` mediante la instrucción `append`, asegurando que la "huella operativa" y la diversificación de los clubes queden registradas de forma permanente en el modelo relacional para su posterior análisis de negocio.

In [ ]:
# 1. Función estandarizada de carga segura
# Implementamos la misma estructura de control de errores (Try-Except) utilizada en las ingestas previas.
# Esto garantiza que si ocurre un problema de red o de base de datos, el flujo se detenga de forma controlada.
def cargar_tabla_detalle(df, table_name, engine):
    print(f"Iniciando carga de tabla: {table_name}...")
    try:
        # if_exists='append': Agrega los nuevos registros sin borrar ni alterar los datos previos de la tabla.
        # index=False: Evita que el índice numérico autogenerado por pandas intente insertarse como columna.
        df.to_sql(name=table_name, con=engine, if_exists='append', index=False)
        print(f"  -> Éxito: {len(df)} registros insertados en dbo.{table_name}.\n")
    except Exception as e:
        print(f"  -> ERROR CRÍTICO al cargar {table_name}:")
        print(e)
        print("\n")

# 2. Ejecución de la carga para la tabla de enriquecimiento
# Inyectamos el DataFrame limpio con las relaciones Club-Videojuego obtenidas de Wikipedia
cargar_tabla_detalle(df_insertar, 'Club_Divisions', engine)

# 3. Mensaje de finalización del pipeline de Web Scraping
print("¡ETL de Tablas de Detalle completado con éxito! La Fase 3 (Preparación de datos) original está terminada.")

Iniciando carga de tabla: Club_Divisions...
  -> Éxito: 69 registros insertados en dbo.Club_Divisions.

¡ETL de Tablas de Detalle completado con éxito! La Fase 3 (Preparación de datos) original está terminada.
